# Интерпретация

In [49]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder, OrdinalEncoder
from sklearn.inspection import partial_dependence, PartialDependenceDisplay, permutation_importance
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import lime
import lime.lime_tabular

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = 'retina'


### Загрузка и подготовка данных


Вам будет предоставлен датасет, посвященный продаже недвижимости. Ваша задача - построить интерпретацию для этого датасета. В нем достаточно много различных признаков, поэтому вы можете предварительно отфильтровать их, когда будете строить графики. Оставляйте достаточно признаков, чтобы ваши модели оставались точными..

In [19]:
data_path = '/home/sonnet/projects/nikolskaya_ml_2026/data.csv'
data = pd.read_csv(data_path, sep=',')

print(f"Размер датасета: {data.shape}")
print(f"\nПервые строки:")
data.head()


Размер датасета: (29905, 83)

Первые строки:


,region_name_cat,district_cat,corpus_cat,developer_cat,agreement_date,floor,square,rooms_4,location_logs_count_mean,location_depth,...,location_public_transport_platform_w_mean_distance,location_water_w_mean_distance,location_university_w_mean_distance,location_leisure_w_mean_distance,location_pop_shop_cnt,price_target,hc_name_cat,interior_cat,class_cat,stage_cat
0,Город,45,538,18,2012-08-10,3.0,62.23,2,22.550466,13.0,...,0.910028,0.782675,-999.000000,0.820073,16.0,28417.424671,50,49786.0,27353,7983
1,Пригород,48,432,63,2013-05-19,11.0,22.52,студия,22.581858,13.0,...,0.902510,0.902673,-999.000000,0.990908,18.0,16728.215463,293,49786.0,97865,70661
2,Город,44,2372,126,2012-12-12,3.0,38.17,1,20.191250,13.0,...,0.851637,-999.000000,-999.000000,0.945618,7.0,18311.834458,284,49786.0,97865,70661
3,Город,14,1053,121,2012-12-10,10.0,57.48,2,23.286900,13.0,...,0.913797,1.028386,0.300026,0.828147,5.0,25171.489968,325,0.0,97865,12638
4,Город,63,2426,69,2012-02-12,3.0,41.43,1,20.599150,13.0,...,1.051049,-999.000000,-999.000000,0.991506,4.0,27324.795343,182,49786.0,97865,70661


In [20]:
data["agreement_date"].value_counts()

agreement_date
2012-12-30    475
2013-03-28    275
2013-05-18    268
2013-03-17    235
2012-12-07    233
             ... 
2012-01-05      2
2012-01-01      1
2013-01-05      1
2012-01-09      1
2012-01-04      1
Name: count, Length: 529, dtype: int64

## Задание 1. 1 балл
Сделайте 2 версии данных - с нормализацией признаков и без.
Обучите 6 моделей:
- линейную регрессию (LinearRegression) на двух вариантах данных
- Lasso регрессию (Lasso) на двух вариантах данных
- градиентный бустинг (GradientBoostingRegressor) на двух вариантах данных. Ограничьте глубину до 5.

Выведите MSE, RMSE и MAPE моделей. Какая функция больше подходит? Почему?

Зафиксируйте выводы. Какие модели чувствительны к масштабу признаков, а какие почти инвариантны? Почему это важно для анализа признаков?

In [47]:
scalers = {
    "standart": StandardScaler(),
    "minmax": MinMaxScaler(),
    "robust": RobustScaler(),
}
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

In [22]:
data = data.drop(columns=["agreement_date"])

In [39]:
cat_columns = ["region_name_cat", "district_cat", "corpus_cat", "developer_cat", "hc_name_cat", "interior_cat", "class_cat", "stage_cat", "rooms_4"]
num_columns = [col for col in data.drop(columns="price_target").columns if col not in cat_columns]
data_cat = data[cat_columns]
data_num = data.loc[:, ~data.columns.isin(cat_columns)]

In [55]:
regressors = {
    "linear": LinearRegression(),
    "lasso": Lasso(),
    "bossting": GradientBoostingRegressor(max_depth=5, random_state=11),
}

In [60]:
for scale_key in scalers:
    data_1 = data.copy().dropna()
    y = data_1["price_target"]
    X = data_1.drop(columns=["price_target"])
    
    X_train, X_test, y_trtain, y_test = train_test_split(X, y, test_size=0.25, random_state=11)

    for reg_key in regressors:
        pipeline = Pipeline([
            ("prep", ColumnTransformer([
                # ("num", scalers[scale_key], num_columns),
                ("cat", encoder, cat_columns)
            ])),
            ("model", regressors[reg_key])
        ])

        pipeline.fit(X_train, y_trtain)
        y_pred = pipeline.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mape = mean_absolute_percentage_error(y_test, y_pred)

        # print(f"scaler: {scale_key}, model: {reg_key}, MSE: {mse:.2f}, RMSE: {rmse:.2f}, MAPE: {mape*100:.2f}%")
        print(f"model: {reg_key}, MSE: {mse:.2f}, RMSE: {rmse:.2f}, MAPE: {mape*100:.2f}%")


        


model: linear, MSE: 40996263.49, RMSE: 6402.83, MAPE: 16.33%
model: lasso, MSE: 40995894.83, RMSE: 6402.80, MAPE: 16.32%
model: bossting, MSE: 4305533.78, RMSE: 2074.98, MAPE: 5.00%
model: linear, MSE: 40996263.49, RMSE: 6402.83, MAPE: 16.33%
model: lasso, MSE: 40995894.83, RMSE: 6402.80, MAPE: 16.32%
model: bossting, MSE: 4305533.78, RMSE: 2074.98, MAPE: 5.00%
model: linear, MSE: 40996263.49, RMSE: 6402.83, MAPE: 16.33%
model: lasso, MSE: 40995894.83, RMSE: 6402.80, MAPE: 16.32%
model: bossting, MSE: 4305533.78, RMSE: 2074.98, MAPE: 5.00%


### Model with scaling result:
scaler: standart, model: linear, MSE: 17136441.16, RMSE: 4139.62, MAPE: 9.97% \
scaler: standart, model: lasso, MSE: 18155979.86, RMSE: 4260.98, MAPE: 10.34% \
scaler: standart, model: bossting, MSE: 3171829.30, RMSE: 1780.96, MAPE: 4.16% \
scaler: minmax, model: linear, MSE: 17136441.16, RMSE: 4139.62, MAPE: 9.97% \
scaler: minmax, model: lasso, MSE: 18275044.73, RMSE: 4274.93, MAPE: 10.34% \
scaler: minmax, model: bossting, MSE: 3177165.42, RMSE: 1782.46, MAPE: 4.20% \
scaler: robust, model: linear, MSE: 17136441.16, RMSE: 4139.62, MAPE: 9.97% \
scaler: robust, model: lasso, MSE: 18164701.82, RMSE: 4262.01, MAPE: 10.34% \
scaler: robust, model: bossting, MSE: 3154545.02, RMSE: 1776.10, MAPE: 4.15% 


### Models without scaling result:
model: linear, MSE: 40996263.49, RMSE: 6402.83, MAPE: 16.33% \
model: lasso, MSE: 40995894.83, RMSE: 6402.80, MAPE: 16.32% \
model: bossting, MSE: 4305533.78, RMSE: 2074.98, MAPE: 5.00% \
model: linear, MSE: 40996263.49, RMSE: 6402.83, MAPE: 16.33% \
model: lasso, MSE: 40995894.83, RMSE: 6402.80, MAPE: 16.32% \
model: bossting, MSE: 4305533.78, RMSE: 2074.98, MAPE: 5.00% \
model: linear, MSE: 40996263.49, RMSE: 6402.83, MAPE: 16.33% \
model: lasso, MSE: 40995894.83, RMSE: 6402.80, MAPE: 16.32% \
model: bossting, MSE: 4305533.78, RMSE: 2074.98, MAPE: 5.00% 


## Задание 1.1(*) 1 балл
Сравните модели, построенные с помощью разных видов нормализации (MinMax, Standart). Отличается ли важность признаков?

In [ ]:
### ваш код

## Задание 2. 1 балл
Выберите 1 признак для анализа (можно категориальный, с не менее чем 5 уровнями, или дискретизируйте непрерывный). 
Используйте линейную регрессию и бустинг после применения MinMaxScaler. Что будет с моделями, если признаки выйдут из диапазона?
Постройте графики ICE и PDP для интерпретации исходных данных, а также искусственно добавив несколько выбросов, выходящих за оригинальные интервалы. 

Задание 2.1 (*) 1 балл: проанализируйте также еще один признак

## Задание 3. 1 балл
Выберите 20 объектов из тестовой выборки.
Для каждого объекта из выбранного набора построим траекторию изменения предсказания модели при постепенном изменении значения признака от его текущего значения к базовому значению (медиана или среднее по обучающей выборке).

**Алгоритм:**
1. Выбрать объект $x_i$ из тестовой выборки
2. Для интересующего признака $j$:
   - Текущее значение: $x_{i,j}$
   - Базовое значение: $x_{base,j}$ (медиана или среднее по обучающей выборке)
3. Построить линейную интерполяцию между $x_{i,j}$ и $x_{base,j}$ с $n$ шагами
4. Для каждого шага интерполяции:
   - Заменить значение признака $j$ в объекте $x_i$ на значение из интерполяции
   - Вычислить предсказание модели для модифицированного объекта
5. Построить график траектории


Задание 3.1 (*) 1 балл: проанализируйте также еще один признак

In [ ]:
### ваш код

## Задание 4 (1 балл). ALE
Постройте ALE по обеим моделям, используя pyALE. Подберите размер сетки так, чтобы получить доверительные интервалы. Проанализируйте полученный график. Каковы получились доверительные интервалы? Почему они различны для моделей?

P.s. Сетку значений стройте для исходного признака.

In [ ]:
### ваш код

## Задание 5:  Permutation Importance (2 балла)
Постройте Permutation importances по обеим моделям, используя sklearn.

Поэкспериментируйте с числом перестановок.

Проанализируйте полученные коэффициенты. Как они меняются от количества перестановок? Как меняются std коэффициентов?



In [2]:
### ваш код

## Задание 5: Feature Importance (2 балла)
Пусть важность - это MAPE для тестовых данных. Проведите анализ только для бустинга

Идея перестановочной важности представляет собой частный случай важности при помощи внесения возмущений в признак. Примеры возмущений:
1) внесение случайного шума
2) зануление признака
3) сдвиг признака к его базовому значению и оценка траектории изменения прогнозов или качества модели 

Примем за базовое значение (${base}$)медиану признака и будем сдвигать исходный признак к медианному с некоторым коэффициентом $\beta$:
$x_j^\beta = (1- \beta)x_j + \beta {base}$

Реализуйте это возмущение. Как меняются важности при разных $\beta$?

Постройте графики важности и сравните важности с permutation importance. Используйте только числовые признаки. При этом медиану стоит считать на тренировочном наборе, а важность как разницу MAPE на тестовой выборке. Чем больше разница, тем важнее признак. 


Сравните результаты методов. Какие признаки наиболее важны? Есть ли различия между методами? В чём могут быть причины различий?

#  Задание 6. 2 балла. LIME.
Постройте интерпретацию признаков для нескольких примеров с помощью LIME. Можете использовать 
Оцените устойчивость реализации. Как влияет на коэффициенты количество сгенерированных точек? А выбор признаков (lasso/добавление фичей по порядку). А выбор ядра?

(*) Вы получите на 2 балла больше, если используете свою реализацию из задания семинарского ноутбука. В таком случае не забудьте добавить тесты для своей реализации. 



## Задание 7. 1 балл. SHAP
Постройте локальный график с SHAP для объекта с индексом, равным вашему номеру в таблице курса на обеих моделях и сделайте выводы. 
## Задание 7.1 (*). 1 балл.  Shap и категориальные переменные.
Shap разлагает предсказание модели вблизи точки x на базовый уровень и сумму вкладов признаков: $ f(x) = base + \sum_i{\phi_i(x)}$. В случае one-hot вклад признака - это сумма вкладов dummy столбцов. Сравните вклады категориальных признаков до и после кодировки - так ли это? 
